# Dataset festivos

Este codigo crea un dataframe de los dias festivos y vacaciones escolares entre los años 2015 y 2027.

## 0. Importación de paquetes

In [ ]:
import pandas as pd
import holidays

## 1. Desarga de datos

In [2]:
# Creamos un calendario con todas las fechas entre 2015 y 2027

calendar = pd.DataFrame({"date": pd.date_range("2015-01-01", "2027-12-31", freq="D")})

In [4]:
# fiestas escolares
school = pd.read_csv("data/vacation_days.csv")


In [7]:
# renombramos columna
school.columns = ["school_holiday"]

# convertimos a tipo fecha
school["school_holiday"] = pd.to_datetime(school["school_holiday"])

# Nos quedamos solo con los años 2015-2027
school = school[
    (school["school_holiday"].dt.year >= 2015) &
    (school["school_holiday"].dt.year <= 2027)
].reset_index(drop=True)

In [9]:
# Insertamos los school holidays en el calendario

# Creamos el nuevo DataFrame
df_holidays = calendar.copy()

df_holidays["school_holiday"] = (
    df_holidays["date"]
    .isin(school["school_holiday"])
    .astype(int)
)

print(df_holidays.head())

        date  school_holiday
0 2015-01-01               1
1 2015-01-02               1
2 2015-01-03               0
3 2015-01-04               0
4 2015-01-05               0


In [ ]:
#descragamos dias festivos

il_holidays = holidays.country_holidays(
    "US",
    subdiv="IL",
    years=range(2015, 2028)  # 2015-2027
)

# creamos df
df_festivos = (
    pd.DataFrame(
        il_holidays.items(),
        columns=["date", "holiday"]
    )
    .sort_values("date")
    .reset_index(drop=True)
)

# Convertir la fecha a datetime
df_festivos["date"] = pd.to_datetime(df_festivos["date"])

df_festivos

,date,holiday
0,2015-01-01,New Year's Day
1,2015-01-19,Martin Luther King Jr. Day
2,2015-02-12,Lincoln's Birthday
3,2015-02-16,Washington's Birthday
4,2015-03-02,Casimir Pulaski Day
...,...,...
186,2027-11-11,Veterans Day
187,2027-11-25,Thanksgiving Day
188,2027-12-24,Christmas Day (observed)
189,2027-12-25,Christmas Day


In [11]:
# lo unimos al df de holidays

# asegurmos de que ambas columnas son datetime
df_holidays["date"] = pd.to_datetime(df_holidays["date"])
df_festivos["date"] = pd.to_datetime(df_festivos["date"])

# crear la columna festivo (1 = festivo, 0 = no festivo)
df_holidays["festivo"] = df_holidays["date"].isin(df_festivos["date"]).astype(int)

In [12]:
df_holidays

,date,school_holiday,festivo
0,2015-01-01,1,1
1,2015-01-02,1,0
2,2015-01-03,0,0
3,2015-01-04,0,0
4,2015-01-05,0,0
...,...,...,...
4743,2027-12-27,0,0
4744,2027-12-28,0,0
4745,2027-12-29,0,0
4746,2027-12-30,0,0


In [13]:
# quitamos fechas mayores que 2027-07-05 porque es hasta donde llegan los datos de school holidays
df_holidays = df_holidays[df_holidays["date"] <= "2027-07-05"].copy()

In [14]:
#convertimos en csv
df_holidays.to_csv("data_clean/holidays.csv", index=False)